In [ ]:
import subprocess, os, zipfile, time, json

# ─── Step 1: Install packages ───
print("=" * 55)
print("  STEP 1: Installing packages...")
print("=" * 55)

subprocess.run([
    "pip", "install", "-q",
    "torchao>=0.16.0",
    "colpali-engine",
    "chromadb",
    "sentence-transformers",
    "accelerate",
    "bitsandbytes",
    "transformers",
    "Pillow",
    "pymupdf",
], check=False)

subprocess.run(
    ["apt-get", "install", "-y", "-q", "poppler-utils"],
    check=False
)
print("  ✅ Packages installed!")

  STEP 1: Installing packages...
  ✅ Packages installed!


In [5]:
# ─── Step 2: Restore files ───
print("\n" + "=" * 55)
print("  STEP 2: Restoring files...")
print("=" * 55)

dirs = [
    "data/raw", "data/parsed/pages",
    "data/parsed/markdown",
    "data/indices/multivectors",
    "data/indices/chroma_index",
    "outputs",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

idx_zip   = "/content/sci-rag-indices.zip"
pages_zip = "/content/sci-rag-pages.zip"

if os.path.exists(idx_zip):
    try:
        with zipfile.ZipFile(idx_zip, "r") as zf:
            zf.extractall("data")
        print(f"  ✅ Indices restored!")
    except zipfile.BadZipFile:
        print(f"  ❌ sci-rag-indices.zip is corrupted or not a valid zip file! Please re-upload it.")
    except Exception as e:
        print(f"  ❌ An unexpected error occurred while extracting sci-rag-indices.zip: {e}")
else:
    print("  ❌ sci-rag-indices.zip missing!")

if os.path.exists(pages_zip):
    try:
        with zipfile.ZipFile(pages_zip, "r") as zf:
            zf.extractall("data")
        print(f"  ✅ Pages restored!")
    except zipfile.BadZipFile:
        print(f"  ❌ sci-rag-pages.zip is corrupted or not a valid zip file! Please re-upload it.")
    except Exception as e:
        print(f"  ❌ An unexpected error occurred while extracting sci-rag-pages.zip: {e}")
else:
    print("  ❌ sci-rag-pages.zip missing!")


  STEP 2: Restoring files...
  ✅ Indices restored!
  ✅ Pages restored!


In [6]:
# ─── Step 3: Verify packages ───
print("\n" + "=" * 55)
print("  STEP 3: Verifying...")
print("=" * 55)

import torch
import numpy as np
import torchao

print(f"  torch          : {torch.__version__}")
print(f"  torchao        : {torchao.__version__}")

if torch.cuda.is_available():
    print(f"  GPU            : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  VRAM total     : {vram:.1f} GB")
    print(f"  VRAM free      : {free:.1f} GB")

try:
    from colpali_engine.models import ColPali, ColPaliProcessor
    print("  colpali-engine : ✅")
except Exception as e:
    print(f"  colpali-engine : ❌ {e}")

try:
    from sentence_transformers import SentenceTransformer
    print("  sentence-trans : ✅")
except Exception as e:
    print(f"  sentence-trans : ❌ {e}")

try:
    import chromadb
    print(f"  chromadb       : {chromadb.__version__} ✅")
except Exception as e:
    print(f"  chromadb       : ❌ {e}")


  STEP 3: Verifying...
  torch          : 2.11.0+cu128
  torchao        : 0.17.0
  GPU            : Tesla T4
  VRAM total     : 14.6 GB
  VRAM free      : 14.5 GB
  colpali-engine : ✅
  sentence-trans : ✅
  chromadb       : 1.5.9 ✅


In [7]:
# ─── Step 4: Load metadata + ChromaDB ───
print("\n" + "=" * 55)
print("  STEP 4: Loading metadata...")
print("=" * 55)

import chromadb

CONFIG = {
    "npy_dir"             : "data/indices/multivectors",
    "chroma_dir"          : "data/indices/chroma_index",
    "pages_dir"           : "data/parsed/pages",
    "colpali_model"       : "vidore/colpali-v1.2",
    "scincl_model"        : "malteos/scincl",
    "qwen_model"          : "Qwen/Qwen2-VL-2B-Instruct",
    "top_k"               : 3,
    "colpali_weight"      : 0.7,
    "scincl_weight"       : 0.3,
    "confidence_threshold": 0.5,
    "max_retries"         : 1,
}

with open("data/indices/page_metadata.json") as f:
    page_metadata = json.load(f)
with open("data/indices/doc_mapping.json") as f:
    doc_mapping = json.load(f)
with open("data/indices/summary.json") as f:
    summary = json.load(f)

print(f"  ✅ page_metadata : {len(page_metadata)} pages")
print(f"  ✅ doc_mapping   : {len(doc_mapping)} papers")

chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_dir"])
collections   = chroma_client.list_collections()
collection    = chroma_client.get_collection(collections[0].name)
print(f"  ✅ ChromaDB      : {collection.count()} entries")

npy_index = {}
for fname in os.listdir(CONFIG["npy_dir"]):
    if fname.endswith(".npy") and not fname.endswith(".meta.npy"):
        page_key = fname.replace(".npy", "")
        npy_index[page_key] = os.path.join(CONFIG["npy_dir"], fname)
print(f"  ✅ npy_index     : {len(npy_index)} files")

print(f"\n{'='*55}")
print(f"  ✅ ALL READY!")
print(f"  GPU  : {torch.cuda.get_device_name(0)}")
print(f"  VRAM : {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB free")
print(f"{'='*55}")
print("✅ RECOVERY CELL COMPLETE — Now run Cell 4 + 5 + 6!")


  STEP 4: Loading metadata...
  ✅ page_metadata : 182 pages
  ✅ doc_mapping   : 10 papers
  ✅ ChromaDB      : 182 entries
  ✅ npy_index     : 182 files

  ✅ ALL READY!
  GPU  : Tesla T4
  VRAM : 14.5 GB free
✅ RECOVERY CELL COMPLETE — Now run Cell 4 + 5 + 6!


In [8]:

# ═══════════════════════════════════════════════════════════════
# CELL 4: Retrieval Functions
# ColPali Visual + SciNCL Text + Score Fusion
# ═══════════════════════════════════════════════════════════════

import gc
import numpy as np
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor
from sentence_transformers import SentenceTransformer

print("=" * 55)
print("  CELL 4: RETRIEVAL FUNCTIONS")
print("=" * 55)

# ═══════════════════════════════════════
# FUNCTION 1: ColPali Visual Retrieval
# ═══════════════════════════════════════

def colpali_retrieve(query, top_k=5):
    """
    Load ColPali → encode query → MaxSim score
    all pages → unload → return top_k results
    """
    print(f"  [ColPali] Loading model...")
    t0 = torch.cuda.mem_get_info()[0] / 1024**3

    # Load
    model = ColPali.from_pretrained(
        CONFIG["colpali_model"],
        torch_dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    processor = ColPaliProcessor.from_pretrained(
        CONFIG["colpali_model"]
    )
    model.eval()

    t1 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  [ColPali] Loaded! VRAM used: {t0-t1:.1f} GB")

    # Encode query
    batch = processor.process_queries(queries=[query])
    batch = {k: v.to(model.device) for k, v in batch.items()}

    with torch.no_grad():
        query_emb = model(**batch)

    query_vec = query_emb[0].cpu().float().numpy()
    del batch, query_emb
    torch.cuda.empty_cache()

    # Score all pages via MaxSim
    print(f"  [ColPali] Scoring {len(npy_index)} pages...")
    scores = []

    for page_key, npy_path in npy_index.items():
        try:
            page_vec   = np.load(npy_path)
            sim_matrix = np.matmul(query_vec, page_vec.T)
            score      = float(sim_matrix.max(axis=1).mean())
            scores.append((page_key, score))
        except Exception as e:
            continue

    scores.sort(key=lambda x: x[1], reverse=True)
    top_scores = scores[:top_k]

    # Unload
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  [ColPali] Done! Top score: {top_scores[0][1]:.4f}")

    # Build results
    results = []
    for page_key, score in top_scores:
        meta = page_metadata.get(page_key, {})
        results.append({
            "page_key"   : page_key,
            "score"      : score,
            "doc_id"     : meta.get("doc_id", ""),
            "page_num"   : meta.get("page_num", 0),
            "paper_title": meta.get("paper_title", ""),
            "image_path" : meta.get("image_path", ""),
            "text"       : meta.get("text", ""),
        })

    return results


# ═══════════════════════════════════════
# FUNCTION 2: SciNCL Text Retrieval
# ═══════════════════════════════════════

def scincl_retrieve(query, top_k=5):
    """
    Load SciNCL → encode query → ChromaDB search
    → unload → return top_k results
    """
    print(f"  [SciNCL] Loading model...")

    # Load
    model = SentenceTransformer(
        CONFIG["scincl_model"],
        device="cuda"
    )

    # Encode
    query_vec = model.encode(
        query,
        convert_to_numpy=True
    )

    # Search ChromaDB
    results_db = collection.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # Unload
    del model
    gc.collect()
    torch.cuda.empty_cache()

    # Build results
    results = []
    for i, doc_id in enumerate(results_db["ids"][0]):
        meta  = results_db["metadatas"][0][i]
        dist  = results_db["distances"][0][i]
        sim   = 1 - dist
        page_meta = page_metadata.get(doc_id, {})
        results.append({
            "page_key"   : doc_id,
            "score"      : sim,
            "doc_id"     : meta.get("doc_id", ""),
            "page_num"   : int(meta.get("page_num", 0)),
            "paper_title": meta.get("paper_title", ""),
            "image_path" : page_meta.get("image_path", ""),
            "text"       : page_meta.get("text", ""),
        })

    print(f"  [SciNCL] Done! Top score: {results[0]['score']:.4f}")
    return results


# ═══════════════════════════════════════
# FUNCTION 3: Score Fusion
# ═══════════════════════════════════════

def fuse_scores(colpali_results, scincl_results,
                colpali_weight=0.7, scincl_weight=0.3,
                top_k=3):
    """
    Normalize + weight + fuse ColPali and SciNCL scores
    Returns top_k fused results
    """

    # Normalize ColPali
    c_scores = {r["page_key"]: r["score"] for r in colpali_results}
    c_vals   = list(c_scores.values())
    c_min, c_max = min(c_vals), max(c_vals)

    # Normalize SciNCL
    s_scores = {r["page_key"]: r["score"] for r in scincl_results}
    s_vals   = list(s_scores.values())
    s_min, s_max = min(s_vals), max(s_vals)

    # Fuse
    fused = {}
    for pk, sc in c_scores.items():
        norm_c    = (sc - c_min) / (c_max - c_min + 1e-8)
        fused[pk] = colpali_weight * norm_c

    for pk, sc in s_scores.items():
        norm_s = (sc - s_min) / (s_max - s_min + 1e-8)
        if pk in fused:
            fused[pk] += scincl_weight * norm_s
        else:
            fused[pk]  = scincl_weight * norm_s

    # Sort
    fused_sorted = sorted(
        fused.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    # Build final results
    all_results = {r["page_key"]: r for r in colpali_results}
    all_results.update({r["page_key"]: r for r in scincl_results})

    final = []
    for page_key, fused_score in fused_sorted:
        r = all_results.get(page_key, {})
        r["fused_score"] = fused_score
        final.append(r)

    return final


# ═══════════════════════════════════════
# FUNCTION 4: Full Retrieve Pipeline
# ═══════════════════════════════════════

def retrieve(query, top_k=3):
    """
    Full retrieval pipeline:
    ColPali + SciNCL + Fusion → top_k pages
    """
    print(f"\n  Query: '{query[:60]}'")
    print(f"  {'─'*50}")

    t_start = time.time()

    # Step 1: ColPali
    colpali_results = colpali_retrieve(query, top_k=10)

    # Step 2: SciNCL
    scincl_results  = scincl_retrieve(query, top_k=10)

    # Step 3: Fusion
    fused_results   = fuse_scores(
        colpali_results,
        scincl_results,
        colpali_weight=CONFIG["colpali_weight"],
        scincl_weight =CONFIG["scincl_weight"],
        top_k=top_k,
    )

    t_end = time.time()

    print(f"\n  [Fusion] Top {top_k} results:")
    for i, r in enumerate(fused_results):
        print(f"    [{i+1}] {r['page_key']}")
        print(f"         {r['paper_title'][:50]}")
        print(f"         fused={r['fused_score']:.4f}")

    print(f"\n  Retrieval time: {t_end-t_start:.1f}s")
    return fused_results

print(f"\n{'='*55}")
print("  ✅ FUNCTIONS LOADED!")
print("  ├── colpali_retrieve(query, top_k)")
print("  ├── scincl_retrieve(query, top_k)")
print("  ├── fuse_scores(c_results, s_results)")
print("  └── retrieve(query, top_k)  ← main function")
print(f"{'='*55}")
print("✅ CELL 4 COMPLETE!")

  CELL 4: RETRIEVAL FUNCTIONS

  ✅ FUNCTIONS LOADED!
  ├── colpali_retrieve(query, top_k)
  ├── scincl_retrieve(query, top_k)
  ├── fuse_scores(c_results, s_results)
  └── retrieve(query, top_k)  ← main function
✅ CELL 4 COMPLETE!


In [3]:

# ═══════════════════════════════════════════════════════════════
# CELL 5: Qwen2-VL Answer Generation Function
# ═══════════════════════════════════════════════════════════════

import torch
import gc
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("=" * 55)
print("  CELL 5: QWEN2-VL GENERATION FUNCTION")
print("=" * 55)

# ═══════════════════════════════════════
# FUNCTION: Generate Answer with Qwen2-VL
# ═══════════════════════════════════════

def generate_answer(query, retrieved_pages):
    """
    Load Qwen2-VL → build prompt with images + text
    → generate answer → unload → return answer
    """

    print(f"  [Qwen2-VL] Loading model...")
    t0 = torch.cuda.mem_get_info()[0] / 1024**3

    # ─── Load Qwen2-VL ───
    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        CONFIG["qwen_model"],
        torch_dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    qwen_processor = AutoProcessor.from_pretrained(
        CONFIG["qwen_model"]
    )
    qwen_model.eval()

    t1 = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  [Qwen2-VL] Loaded! VRAM used: {t0-t1:.1f} GB")

    # ─── Build context from retrieved pages ───
    context_texts = []
    context_images = []

    for i, page in enumerate(retrieved_pages[:3]):
        # Add text context
        text = page.get("text", "").strip()[:400]
        title = page.get("paper_title", "")
        page_num = page.get("page_num", 0)

        context_texts.append(
            f"[Source {i+1}] {title} — Page {page_num}:\n{text}"
        )

        # Add image if exists
        img_path = page.get("image_path", "")
        if img_path and os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert("RGB")
                # Resize to save VRAM
                img = img.resize((512, 512), Image.LANCZOS)
                context_images.append(img)
            except:
                pass

    context_str = "\n\n".join(context_texts)

    print(f"  [Qwen2-VL] Context: {len(context_images)} images, "
          f"{len(context_texts)} text chunks")

    # ─── Build messages ───
    # Build content list with images first then text
    content = []

    for img in context_images:
        content.append({
            "type" : "image",
            "image": img,
        })

    content.append({
        "type": "text",
        "text": f"""You are a scientific paper assistant.
Answer the question based on the provided paper pages.
Be specific and cite which paper/page supports your answer.

Context from papers:
{context_str}

Question: {query}

Answer:"""
    })

    messages = [
        {
            "role"   : "user",
            "content": content,
        }
    ]

    # ─── Process + Generate ───
    try:
        text_input = qwen_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # Get images list for processor
        images_for_proc = context_images if context_images else None

        inputs = qwen_processor(
            text=text_input,
            images=images_for_proc,
            return_tensors="pt",
        ).to(qwen_model.device)

        print(f"  [Qwen2-VL] Generating answer...")

        with torch.no_grad():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                temperature=1.0,
                repetition_penalty=1.1,
            )

        # Decode only new tokens
        input_len = inputs["input_ids"].shape[1]
        new_tokens = output_ids[0][input_len:]
        answer = qwen_processor.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

    except Exception as e:
        print(f"  [Qwen2-VL] ⚠️ Error: {e}")
        answer = f"Error generating answer: {e}"

    finally:
        # ─── Unload Qwen2-VL ───
        print(f"  [Qwen2-VL] Unloading...")
        del qwen_model, qwen_processor
        gc.collect()
        torch.cuda.empty_cache()
        freed = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"  [Qwen2-VL] Unloaded! VRAM free: {freed:.1f} GB")

    return answer


# ═══════════════════════════════════════
# FULL RAG PIPELINE FUNCTION
# ═══════════════════════════════════════

def rag_query(query):
    """
    Complete RAG Pipeline:
    Query → Retrieve → Generate → Return
    """
    print(f"\n{'='*55}")
    print(f"  RAG QUERY")
    print(f"{'='*55}")
    print(f"  Q: {query}")
    print(f"{'='*55}")

    t_start = time.time()

    # ─── Step 1: Retrieve ───
    print(f"\n  STEP 1: RETRIEVAL")
    retrieved = retrieve(query, top_k=CONFIG["top_k"])

    # ─── Step 2: Generate ───
    print(f"\n  STEP 2: GENERATION")
    answer = generate_answer(query, retrieved)

    t_total = time.time() - t_start

    # ─── Step 3: Build output ───
    sources = []
    for r in retrieved:
        sources.append({
            "paper"      : r.get("paper_title", "")[:50],
            "page"       : r.get("page_num", 0),
            "arxiv_id"   : r.get("doc_id", ""),
            "fused_score": round(r.get("fused_score", 0), 4),
        })

    result = {
        "query"      : query,
        "answer"     : answer,
        "sources"    : sources,
        "total_time" : round(t_total, 1),
    }

    # ─── Print result ───
    print(f"\n{'='*55}")
    print(f"  ✅ ANSWER ({t_total:.1f}s)")
    print(f"{'='*55}")
    print(f"\n  {answer}")
    print(f"\n  📚 Sources:")
    for s in sources:
        print(f"    - {s['paper']}")
        print(f"      Page {s['page']} | "
              f"arxiv:{s['arxiv_id']} | "
              f"score:{s['fused_score']}")
    print(f"{'='*55}")

    return result

print(f"\n{'='*55}")
print("  ✅ GENERATION FUNCTIONS LOADED!")
print("  ├── generate_answer(query, pages)")
print("  └── rag_query(query)  ← USE THIS!")
print(f"{'='*55}")
print("✅ CELL 5 COMPLETE!")

  CELL 5: QWEN2-VL GENERATION FUNCTION

  ✅ GENERATION FUNCTIONS LOADED!
  ├── generate_answer(query, pages)
  └── rag_query(query)  ← USE THIS!
✅ CELL 5 COMPLETE!


In [9]:

# ═══════════════════════════════════════════════════════════════
# CELL 6: FIRST REAL RAG TEST QUERY
# Full pipeline: Retrieve + Generate Answer
# Time: ~1-2 minutes
# ═══════════════════════════════════════════════════════════════

print("=" * 55)
print("  CELL 6: FIRST RAG TEST")
print("=" * 55)

# ─── Run first query ───
result = rag_query("what is Vit transformer ")

# ─── Pretty print ───
print(f"\n{'═'*55}")
print(f"  FINAL RESULT SUMMARY")
print(f"{'═'*55}")
print(f"  Query      : {result['query']}")
print(f"  Total time : {result['total_time']}s")
print(f"\n  Answer:")
print(f"  {result['answer']}")
print(f"\n  Sources used:")
for i, s in enumerate(result['sources']):
    print(f"  [{i+1}] {s['paper']}")
    print(f"       Page {s['page']} | "
          f"https://arxiv.org/abs/{s['arxiv_id']} | "
          f"score: {s['fused_score']}")
print(f"{'═'*55}")
print("✅ CELL 6 COMPLETE!")

  CELL 6: FIRST RAG TEST

  RAG QUERY
  Q: what is Vit transformer 

  STEP 1: RETRIEVAL

  Query: 'what is Vit transformer '
  ──────────────────────────────────────────────────
  [ColPali] Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/78.6M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

  [Qwen2-VL] Loaded! VRAM used: 0.1 GB
  [Qwen2-VL] Context: 3 images, 3 text chunks
  [Qwen2-VL] Generating answer...
  [Qwen2-VL] Unloading...
  [Qwen2-VL] Unloaded! VRAM free: 9.0 GB

  ✅ ANSWER (624.6s)

  The Vit transformer is a type of neural network used for vision tasks. It has recently emerged as a competitive alternative to traditional convolutional neural networks (CNNs), which are ubiquitous across the field of computer vision. The Vit transformer's main advantage is its ability to perform well with large amounts of training data, even when the number of edges in the graph is small. This makes it particularly useful for applications such as image classification, object detection, and semantic image segmentation.

  📚 Sources:
    - DeepViT: Towards Deeper Vision Transformer
      Page 2 | arxiv:2108.00102 | score:0.7
    - Training data-efficient image transformers & disti
      Page 1 | arxiv:2106.10270 | score:0.5701
    - DeepViT: Towards Deeper Vision Transformer
     

In [10]:

# ═══════════════════════════════════════════════════════════════
# CELL 7: Domain Guard
# Filters out-of-scope questions BEFORE hitting LLM
# ═══════════════════════════════════════════════════════════════

print("=" * 55)
print("  CELL 7: DOMAIN GUARD")
print("=" * 55)

def is_relevant_query(query):
    """
    Returns (is_relevant: bool, confidence: float, matched: list)
    confidence = how many keywords matched / threshold
    """
    domain_keywords = [
        "transformer", "vit", "vision", "attention", "patch",
        "image", "model", "paper", "architecture", "dataset",
        "training", "accuracy", "classification", "embedding",
        "token", "layer", "encoder", "pretrain", "finetune",
        "swin", "deit", "cnn", "resnet", "efficientformer",
        "multi-head", "self-attention", "positional encoding",
        "benchmark", "imagenet", "inference", "parameter",
        "deep learning", "neural", "weight", "gradient",
        "convolution", "pooling", "softmax", "mlp",
        "scale", "efficient", "distillation", "segmentation",
        "colapali", "scincl", "qwen", "rag", "retrieval",
        "figure", "table", "results", "experiment", "baseline",
        "performance", "memory", "latency", "throughput",
        "head", "block", "depth", "width", "resolution",
        "class", "label", "feature", "representation",
    ]

    query_lower  = query.lower()
    matched      = [kw for kw in domain_keywords if kw in query_lower]
    match_count  = len(matched)

    # Confidence scoring
    # 0 matches   → 0.0  (fully out of scope)
    # 1 match     → 0.35 (borderline)
    # 2+ matches  → 0.6+ (likely in scope)
    if match_count == 0:
        confidence = 0.0
        relevant   = False
    elif match_count == 1:
        confidence = 0.35
        relevant   = True   # allow but warn
    else:
        confidence = min(0.6 + (match_count - 2) * 0.08, 1.0)
        relevant   = True

    return relevant, round(confidence, 2), matched


# ─── Quick Tests ───
test_queries = [
    "What is the capital of France?",
    "How does patch embedding work in ViT?",
    "Who won the cricket world cup?",
    "What is the attention mechanism?",
    "What is 2 + 2?",
    "How does Swin Transformer handle resolution?",
]

print("\n  Guard Test Results:")
print(f"  {'Query':<45} {'Relevant':<10} {'Conf':<8} {'Matched'}")
print(f"  {'─'*45} {'─'*10} {'─'*8} {'─'*20}")

for q in test_queries:
    rel, conf, matched = is_relevant_query(q)
    status = "✅ IN SCOPE" if rel else "🚫 BLOCKED"
    print(f"  {q:<45} {status:<10} {conf:<8} {matched[:2]}")

print(f"\n✅ CELL 7 COMPLETE — Domain Guard Ready!")

  CELL 7: DOMAIN GUARD

  Guard Test Results:
  Query                                         Relevant   Conf     Matched
  ───────────────────────────────────────────── ────────── ──────── ────────────────────
  What is the capital of France?                🚫 BLOCKED  0.0      []
  How does patch embedding work in ViT?         ✅ IN SCOPE 0.68     ['vit', 'patch']
  Who won the cricket world cup?                🚫 BLOCKED  0.0      []
  What is the attention mechanism?              ✅ IN SCOPE 0.35     ['attention']
  What is 2 + 2?                                🚫 BLOCKED  0.0      []
  How does Swin Transformer handle resolution?  ✅ IN SCOPE 0.68     ['transformer', 'swin']

✅ CELL 7 COMPLETE — Domain Guard Ready!


In [11]:

# ═══════════════════════════════════════════════════════════════
# CELL 8: Strict RAG Generation Function
# Qwen2-VL — Only answers from PDF context
# ═══════════════════════════════════════════════════════════════

import gc, os, time, torch
import numpy as np
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

print("=" * 55)
print("  CELL 8: STRICT GENERATION FUNCTION")
print("=" * 55)


def generate_strict(query, retrieved_pages):
    """
    Strict Qwen2-VL RAG generation.
    Returns: {
        answer         : str,
        retrieval_conf : float,   ← from fused scores
        is_from_docs   : bool,
        sources        : list
    }
    """
    print(f"  [Qwen2-VL] Loading...")

    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        CONFIG["qwen_model"],
        torch_dtype=torch.float16,
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    qwen_processor = AutoProcessor.from_pretrained(CONFIG["qwen_model"])
    qwen_model.eval()

    free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"  [Qwen2-VL] Loaded! VRAM free: {free:.1f} GB")

    # ─── Build context + sources ───
    context_texts  = []
    context_images = []
    sources        = []

    for i, page in enumerate(retrieved_pages[:3]):
        text         = page.get("text", "").strip()[:600]
        title        = page.get("paper_title", "Unknown")
        page_num     = page.get("page_num", 0)
        doc_id       = page.get("doc_id", "")
        fused_score  = page.get("fused_score", 0.0)

        context_texts.append(
            f"[Paper {i+1}: {title} | arXiv:{doc_id} | Page {page_num}]\n{text}"
        )

        sources.append({
            "paper_title" : title,
            "arxiv_id"    : doc_id,
            "page_num"    : page_num,
            "fused_score" : round(fused_score, 4),
            "arxiv_url"   : f"https://arxiv.org/abs/{doc_id}",
            "text_snippet": text[:200],
        })

        img_path = page.get("image_path", "")
        if img_path and os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize((448, 448), Image.LANCZOS)
                context_images.append(img)
            except:
                pass

    context_str    = "\n\n---\n\n".join(context_texts)
    retrieval_conf = retrieved_pages[0].get("fused_score", 0.0) if retrieved_pages else 0.0

    # ─── STRICT PROMPT ───
    prompt = f"""You are a STRICT document Q&A assistant.

STRICT RULES:
1. ONLY use information from the CONTEXT PAPERS below
2. NEVER use your own training knowledge
3. If the answer is NOT found in context, say EXACTLY:
   "NOT_IN_DOCUMENTS: This question is not covered in the provided papers."
4. Always cite: paper name + page number for every claim
5. Be technical and specific

CONTEXT FROM PAPERS:
{context_str}

QUESTION: {query}

ANSWER (from papers only — cite paper + page for every point):"""

    # ─── Messages ───
    content = []
    for img in context_images:
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    try:
        text_input = qwen_processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = qwen_processor(
            text=text_input,
            images=context_images if context_images else None,
            return_tensors="pt",
        ).to(qwen_model.device)

        print(f"  [Qwen2-VL] Generating...")

        with torch.no_grad():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=450,
                do_sample=False,
                repetition_penalty=1.1,
            )

        input_len  = inputs["input_ids"].shape[1]
        new_tokens = output_ids[0][input_len:]
        answer     = qwen_processor.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        # Check if model flagged as not in docs
        is_from_docs = "NOT_IN_DOCUMENTS" not in answer
        if not is_from_docs:
            answer = answer.replace("NOT_IN_DOCUMENTS:", "").strip()

    except Exception as e:
        print(f"  ❌ Error: {e}")
        answer       = f"Generation error: {e}"
        is_from_docs = False

    finally:
        del qwen_model, qwen_processor
        gc.collect()
        torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"  [Qwen2-VL] Unloaded! VRAM free: {free:.1f} GB")

    return {
        "answer"        : answer,
        "retrieval_conf": round(retrieval_conf, 4),
        "is_from_docs"  : is_from_docs,
        "sources"       : sources,
    }


# ─── Master RAG function used by UI ───
def full_rag_pipeline(query):
    """
    Master function called by Gradio UI
    Handles everything: guard → retrieve → generate
    Returns structured result for UI rendering
    """
    t_start = time.time()

    # Step 1: Domain Guard
    relevant, guard_conf, matched_kws = is_relevant_query(query)

    if not relevant:
        return {
            "query"         : query,
            "answer"        : "This question is not covered in the provided research papers.",
            "out_of_scope"  : True,
            "guard_conf"    : guard_conf,
            "retrieval_conf": 0.0,
            "final_conf"    : 0.0,
            "is_from_docs"  : False,
            "sources"       : [],
            "matched_kws"   : matched_kws,
            "total_time"    : round(time.time() - t_start, 1),
        }

    # Step 2: Retrieve
    retrieved = retrieve(query, top_k=CONFIG["top_k"])

    if not retrieved or retrieved[0].get("fused_score", 0) < 0.08:
        return {
            "query"         : query,
            "answer"        : "No sufficiently relevant pages found in the documents for this query.",
            "out_of_scope"  : True,
            "guard_conf"    : guard_conf,
            "retrieval_conf": 0.0,
            "final_conf"    : 0.0,
            "is_from_docs"  : False,
            "sources"       : [],
            "matched_kws"   : matched_kws,
            "total_time"    : round(time.time() - t_start, 1),
        }

    # Step 3: Generate
    gen_result = generate_strict(query, retrieved)

    # Step 4: Final confidence score
    # Blend guard confidence + retrieval confidence
    final_conf = round(
        0.3 * guard_conf + 0.7 * gen_result["retrieval_conf"], 4
    ) if gen_result["is_from_docs"] else round(guard_conf * 0.2, 4)

    return {
        "query"         : query,
        "answer"        : gen_result["answer"],
        "out_of_scope"  : not gen_result["is_from_docs"],
        "guard_conf"    : guard_conf,
        "retrieval_conf": gen_result["retrieval_conf"],
        "final_conf"    : final_conf,
        "is_from_docs"  : gen_result["is_from_docs"],
        "sources"       : gen_result["sources"],
        "matched_kws"   : matched_kws,
        "total_time"    : round(time.time() - t_start, 1),
    }


print(f"\n{'='*55}")
print("  ✅ FUNCTIONS READY!")
print("  ├── generate_strict(query, pages)")
print("  └── full_rag_pipeline(query)  ← used by UI")
print(f"{'='*55}")
print("✅ CELL 8 COMPLETE!")

  CELL 8: STRICT GENERATION FUNCTION

  ✅ FUNCTIONS READY!
  ├── generate_strict(query, pages)
  └── full_rag_pipeline(query)  ← used by UI
✅ CELL 8 COMPLETE!


In [12]:

# ═══════════════════════════════════════════════════════════════
# CELL 9: Gradio UI — Beautiful Q&A Interface
# Full RAG UI with confidence + sources + reference links
# ═══════════════════════════════════════════════════════════════

import gradio as gr
import time

print("=" * 55)
print("  CELL 9: GRADIO UI")
print("=" * 55)

# ─── Confidence bar helper ───
def make_confidence_bar(score, label="Confidence"):
    pct = int(score * 100)
    if pct >= 70:
        color = "#22c55e"
        emoji = "🟢"
    elif pct >= 40:
        color = "#f59e0b"
        emoji = "🟡"
    else:
        color = "#ef4444"
        emoji = "🔴"

    bar = f"""
    <div style="margin: 6px 0;">
        <div style="display:flex; justify-content:space-between; margin-bottom:3px;">
            <span style="font-weight:600; color:#374151;">{emoji} {label}</span>
            <span style="font-weight:700; color:{color};">{pct}%</span>
        </div>
        <div style="background:#e5e7eb; border-radius:999px; height:10px;">
            <div style="background:{color}; width:{pct}%;
                        height:10px; border-radius:999px;"></div>
        </div>
    </div>
    """
    return bar


# ─── Source card helper ───
def make_source_card(source, rank):
    score_pct = int(source["fused_score"] * 100)
    snippet   = source["text_snippet"].replace("\n", " ")[:180]

    if score_pct >= 70:
        border = "#22c55e"
    elif score_pct >= 40:
        border = "#f59e0b"
    else:
        border = "#ef4444"

    card = f"""
    <div style="border-left: 4px solid {border};
                background: #f9fafb;
                border-radius: 8px;
                padding: 12px 16px;
                margin: 10px 0;
                box-shadow: 0 1px 3px rgba(0,0,0,0.08);">

        <div style="display:flex; justify-content:space-between; align-items:center;">
            <span style="font-weight:700; color:#1e40af; font-size:14px;">
                📄 [{rank}] {source['paper_title'][:55]}
            </span>
            <span style="background:{border}; color:white;
                         padding:2px 10px; border-radius:999px;
                         font-size:12px; font-weight:700;">
                {score_pct}% match
            </span>
        </div>

        <div style="margin-top:6px; color:#6b7280; font-size:12px;">
            📖 Page {source['page_num']} &nbsp;|&nbsp;
            🔗 <a href="{source['arxiv_url']}" target="_blank"
                  style="color:#3b82f6; text-decoration:none;">
                arXiv:{source['arxiv_id']}
            </a>
        </div>

        <div style="margin-top:8px; color:#374151;
                    font-size:13px; font-style:italic;
                    border-top:1px solid #e5e7eb; padding-top:8px;">
            "{snippet}..."
        </div>
    </div>
    """
    return card


# ─── Out of scope card ───
def make_out_of_scope_card(query, answer):
    card = f"""
    <div style="border: 2px solid #f59e0b;
                background: #fffbeb;
                border-radius: 12px;
                padding: 20px;
                margin: 10px 0;">

        <div style="font-size:18px; font-weight:700;
                    color:#92400e; margin-bottom:10px;">
            ⚠️ Outside Document Scope
        </div>

        <div style="color:#78350f; font-size:14px; margin-bottom:14px;">
            This question was <b>not found</b> in the ingested research papers.
            The answer below comes from general knowledge and
            <b>is NOT sourced from your documents.</b>
        </div>

        <div style="background:#fef3c7; border-radius:8px;
                    padding:14px; color:#451a03; font-size:14px;">
            <b>General Answer:</b><br><br>
            {answer}
        </div>

        <div style="margin-top:12px; color:#b45309;
                    font-size:12px; font-style:italic;">
            💡 Tip: Ask questions about Vision Transformers,
            ViT, attention mechanisms, image patches, or the
            papers in your collection for document-grounded answers.
        </div>
    </div>
    """
    return card


# ─── Answer card ───
def make_answer_card(answer):
    card = f"""
    <div style="border: 2px solid #22c55e;
                background: #f0fdf4;
                border-radius: 12px;
                padding: 20px;
                margin: 10px 0;">

        <div style="font-size:16px; font-weight:700;
                    color:#14532d; margin-bottom:12px;">
            ✅ Answer from Documents
        </div>

        <div style="color:#1a2e1a; font-size:15px;
                    line-height:1.7;">
            {answer}
        </div>
    </div>
    """
    return card


# ═══════════════════════════════════════
# MAIN GRADIO FUNCTION
# ═══════════════════════════════════════

def gradio_rag(query):
    """
    Called by Gradio on every submit.
    Returns: answer_html, confidence_html, sources_html
    """

    if not query or query.strip() == "":
        empty = "<p style='color:#9ca3af;'>Please enter a question.</p>"
        return empty, empty, empty

    # ─── Run full pipeline ───
    result = full_rag_pipeline(query.strip())

    # ═══════════════════════════
    # BUILD: Confidence Panel
    # ═══════════════════════════
    conf_html = f"""
    <div style="background:white; border-radius:12px;
                padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.08);">

        <div style="font-size:15px; font-weight:700;
                    color:#111827; margin-bottom:12px;">
            📊 Confidence Scores
        </div>

        {make_confidence_bar(result['final_conf'],    "Overall Confidence")}
        {make_confidence_bar(result['guard_conf'],    "Domain Relevance")}
        {make_confidence_bar(result['retrieval_conf'],"Retrieval Match")}

        <div style="margin-top:14px; padding:10px;
                    background:#f3f4f6; border-radius:8px;
                    font-size:12px; color:#6b7280;">
            <b>Status:</b>
            {'🟢 Answered from documents' if result['is_from_docs']
              else '🔴 Outside document scope'}
            <br>
            <b>Time:</b> {result['total_time']}s
            <br>
            <b>Keywords matched:</b>
            {', '.join(result['matched_kws'][:5]) if result['matched_kws']
              else 'None'}
        </div>
    </div>
    """

    # ═══════════════════════════
    # BUILD: Answer Panel
    # ═══════════════════════════
    if result["is_from_docs"]:
        answer_html = make_answer_card(result["answer"])
    else:
        answer_html = make_out_of_scope_card(
            result["query"],
            result["answer"]
        )

    # ═══════════════════════════
    # BUILD: Sources Panel
    # ═══════════════════════════
    if result["sources"]:
        sources_inner = "".join([
            make_source_card(src, i + 1)
            for i, src in enumerate(result["sources"])
        ])
        sources_html = f"""
        <div style="background:white; border-radius:12px;
                    padding:16px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.08);">
            <div style="font-size:15px; font-weight:700;
                        color:#111827; margin-bottom:4px;">
                📚 Retrieved Sources ({len(result['sources'])})
            </div>
            <div style="font-size:12px; color:#9ca3af; margin-bottom:12px;">
                Pages retrieved from your ingested PDFs
            </div>
            {sources_inner}
        </div>
        """
    else:
        sources_html = """
        <div style="background:#fef2f2; border-radius:12px;
                    padding:16px; color:#991b1b; font-size:14px;">
            🚫 No relevant pages found in the documents.
        </div>
        """

    return answer_html, conf_html, sources_html


# ═══════════════════════════════════════
# GRADIO UI LAYOUT
# ═══════════════════════════════════════

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Scientific RAG System",
    css="""
        .gradio-container { max-width: 1200px !important; }
        .gr-button-primary {
            background: #2563eb !important;
            border: none !important;
        }
        footer { display: none !important; }
    """
) as demo:

    # ─── Header ───
    gr.HTML("""
    <div style="text-align:center; padding:30px 0 10px 0;">
        <h1 style="font-size:32px; font-weight:800;
                   color:#1e3a5f; margin-bottom:6px;">
            🔬 Scientific RAG System
        </h1>
        <p style="color:#6b7280; font-size:15px;">
            Ask questions about Vision Transformer research papers.
            Powered by ColPali + SciNCL + Qwen2-VL.
        </p>
        <div style="display:flex; justify-content:center;
                    gap:12px; margin-top:12px; flex-wrap:wrap;">
            <span style="background:#dbeafe; color:#1e40af;
                         padding:4px 14px; border-radius:999px;
                         font-size:13px;">📄 10 Papers</span>
            <span style="background:#dcfce7; color:#166534;
                         padding:4px 14px; border-radius:999px;
                         font-size:13px;">🖼️ ColPali Visual</span>
            <span style="background:#fef9c3; color:#854d0e;
                         padding:4px 14px; border-radius:999px;
                         font-size:13px;">📝 SciNCL Text</span>
            <span style="background:#f3e8ff; color:#6b21a8;
                         padding:4px 14px; border-radius:999px;
                         font-size:13px;">🤖 Qwen2-VL</span>
        </div>
    </div>
    """)

    gr.HTML("<hr style='border:1px solid #e5e7eb; margin:0 0 20px 0;'>")

    # ─── Input Row ───
    with gr.Row():
        with gr.Column(scale=5):
            query_box = gr.Textbox(
                label="Ask a Question",
                placeholder=(
                    "e.g. How does patch embedding work in Vision Transformer?"
                ),
                lines=2,
                max_lines=4,
            )
        with gr.Column(scale=1, min_width=120):
            submit_btn = gr.Button(
                "🔍 Ask",
                variant="primary",
                size="lg",
            )
            clear_btn = gr.Button(
                "🗑️ Clear",
                variant="secondary",
                size="sm",
            )

    # ─── Example Questions ───
    gr.Examples(
        examples=[
            ["What is the Vision Transformer and how does it work?"],
            ["How does patch embedding work in ViT?"],
            ["What is the difference between ViT and CNN?"],
            ["How does Swin Transformer handle different resolutions?"],
            ["What datasets were used to train ViT?"],
            ["What is the capital of France?"],
            ["Who won the cricket world cup?"],
        ],
        inputs=query_box,
        label="💡 Example Questions (last 2 are out-of-scope tests)",
    )

    gr.HTML("<hr style='border:1px solid #e5e7eb; margin:20px 0;'>")

    # ─── Output Row ───
    with gr.Row():

        # Left: Answer
        with gr.Column(scale=3):
            gr.HTML("""
            <div style="font-size:14px; font-weight:600;
                        color:#374151; margin-bottom:6px;">
                💬 Answer
            </div>""")
            answer_out = gr.HTML(
                value="<p style='color:#9ca3af; padding:20px;'>"
                      "Your answer will appear here...</p>"
            )

        # Right: Confidence + Sources
        with gr.Column(scale=2):
            gr.HTML("""
            <div style="font-size:14px; font-weight:600;
                        color:#374151; margin-bottom:6px;">
                📊 Confidence
            </div>
            """)
            conf_out = gr.HTML(
                value="<p style='color:#9ca3af; padding:10px;'>"
                      "Confidence scores will appear here...</p>"
            )

            gr.HTML("""
            <div style="font-size:14px; font-weight:600;
                        color:#374151; margin:16px 0 6px 0;">
                📚 Sources & References
            </div>
            """)
            sources_out = gr.HTML(
                value="<p style='color:#9ca3af; padding:10px;'>"
                      "Retrieved sources will appear here...</p>"
            )

    # ─── Footer ───
    gr.HTML("""
    <hr style='border:1px solid #e5e7eb; margin:20px 0 10px 0;'>
    <div style="text-align:center; color:#9ca3af; font-size:12px; padding-bottom:10px;">
        🟢 In-scope answers cite paper + page number &nbsp;|&nbsp;
        🔴 Out-of-scope questions are flagged with zero confidence &nbsp;|&nbsp;
        Built with ColPali · SciNCL · Qwen2-VL
    </div>
 """)

    # ─── Button Actions ───
    submit_btn.click(
        fn=gradio_rag,
        inputs=[query_box],
        outputs=[answer_out, conf_out, sources_out],
    )

    query_box.submit(
        fn=gradio_rag,
        inputs=[query_box],
        outputs=[answer_out, conf_out, sources_out],
    )

    clear_btn.click(
        fn=lambda: (
            "<p style='color:#9ca3af; padding:20px;'>Your answer will appear here...</p>",
            "<p style='color:#9ca3af; padding:10px;'>Confidence scores will appear here...</p>",
            "<p style='color:#9ca3af; padding:10px;'>Retrieved sources will appear here...</p>",
            "",
        ),
        inputs=[],
        outputs=[answer_out, conf_out, sources_out, query_box],
    )


# ═══════════════════════════════════════
# LAUNCH
# ═══════════════════════════════════════

print("\n  Launching Gradio UI...")
print("  ├── In-scope   → green answer + sources + confidence")
print("  ├── Out-of-scope → yellow warning card + 0% confidence")
print("  └── Sources    → paper title + page + arxiv link + snippet")
print()

demo.launch(
    share=True,          # gives public URL
    debug=False,
    show_error=True,
    quiet=False,
)

print("✅ CELL 9 COMPLETE — UI is LIVE!")

  CELL 9: GRADIO UI


/tmp/ipykernel_37060/2490412701.py:247: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_37060/2490412701.py:247: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(



  Launching Gradio UI...
  ├── In-scope   → green answer + sources + confidence
  ├── Out-of-scope → yellow warning card + 0% confidence
  └── Sources    → paper title + page + arxiv link + snippet

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e073448ae048fe7d95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ CELL 9 COMPLETE — UI is LIVE!
